In [ ]:
from liger_kernel.transformers.experimental. matmul
LigerEmbeddingFunction??

In [ ]:
LigerFusedLinearDPOFunction??

In [ ]:
import triton
import triton.language as tl
import torch

# Kernel 1: Compute partial maximum dot products and corresponding indices.
# Grid dimensions: (batch, num_vocab_blocks)
@triton.jit
def partial_max_kernel(x_ptr, weight_ptr, partial_max_ptr, partial_argmax_ptr,
                       batch: tl.constexpr, latent: tl.constexpr, vocab: tl.constexpr,
                       BLOCK_SIZE: tl.constexpr, LATENT_TILE: tl.constexpr):
    # Each kernel instance works on one row of x and one block of vocabulary.
    row = tl.program_id(0)
    block_id = tl.program_id(1)
    # Compute the starting vocabulary index for this block.
    vocab_start = block_id * BLOCK_SIZE
    cur_block = min(BLOCK_SIZE, vocab - vocab_start)  # Use Python's min for scalars
    
    if row >= batch:
        return

    max_val = -1e9
    max_idx = 0

    # Create a constant tile range for latent dimension
    tile_range = tl.arange(0, LATENT_TILE)  # LATENT_TILE is a compile-time constant

    # Loop over the candidates in the current vocabulary block.
    for j in range(0, cur_block):
        candidate = vocab_start + j
        dot = 0.0
        # Loop over the latent dimension in tiles.
        for k in range(0, latent, LATENT_TILE):
            cur_tile = min(LATENT_TILE, latent - k)
            # Use the fixed tile_range and create a mask to ignore out-of-bound indices.
            mask = tile_range < cur_tile
            x_tile = tl.load(x_ptr + row * latent + k + tile_range, mask=mask, other=0.0)
            w_tile = tl.load(weight_ptr + candidate * latent + k + tile_range, mask=mask, other=0.0)
            dot += tl.dot(x_tile, w_tile)
        if dot > max_val:
            max_val = dot
            max_idx = candidate

    num_blocks = (vocab + BLOCK_SIZE - 1) // BLOCK_SIZE
    out_index = row * num_blocks + block_id
    tl.store(partial_max_ptr + out_index, max_val)
    tl.store(partial_argmax_ptr + out_index, max_idx)


# Kernel 2: For each row, reduce partial results to a final prediction and update accuracy.
@triton.jit
def final_reduce_kernel(partial_max_ptr, partial_argmax_ptr, t_ptr, acc_ptr,
                        batch: tl.constexpr, num_blocks: tl.constexpr):
    row = tl.program_id(0)
    if row >= batch:
        return

    final_max = -1e9
    final_argmax = 0
    # Reduce over the vocabulary blocks for the current row.
    for b in range(0, num_blocks):
        index = row * num_blocks + b
        val = tl.load(partial_max_ptr + index)
        arg = tl.load(partial_argmax_ptr + index)
        if val > final_max:
            final_max = val
            final_argmax = arg

    target = tl.load(t_ptr + row)
    correct = tl.where(final_argmax == target, 1, 0)
    tl.atomic_add(acc_ptr, correct)


def compute_accuracy(x: torch.Tensor, weight: torch.Tensor, t: torch.Tensor,
                     BLOCK_SIZE: int = 128, LATENT_TILE: int = 32) -> float:
    """
    Computes accuracy without materializing the full logits matrix.
    
    Args:
      x: Tensor of shape [batch, latent]
      weight: Tensor of shape [vocab, latent]
      t: Tensor of shape [batch] with target indices
      BLOCK_SIZE: Number of vocabulary entries processed per kernel instance.
      LATENT_TILE: Tile size for latent dimension reduction.
    
    Returns:
      Accuracy as a float.
    """
    batch, latent = x.shape
    vocab, latent2 = weight.shape
    assert latent == latent2, "latent dimensions must match"
    
    num_blocks = (vocab + BLOCK_SIZE - 1) // BLOCK_SIZE
    
    partial_max = torch.empty((batch * num_blocks,), device=x.device, dtype=torch.float32)
    partial_argmax = torch.empty((batch * num_blocks,), device=x.device, dtype=torch.int32)
    acc = torch.zeros((1,), device=x.device, dtype=torch.int32)
    
    grid = (batch, num_blocks)
    partial_max_kernel[grid](x, weight, partial_max, partial_argmax,
                               batch, latent, vocab, BLOCK_SIZE, LATENT_TILE)
    
    final_reduce_kernel[(batch,)](partial_max, partial_argmax, t, acc, batch, num_blocks)
    
    correct = acc.item()
    return correct / batch


# Example usage:
if __name__ == '__main__':
    batch = 64
    latent = 128
    vocab = 100000  # Assume a huge vocabulary.
    x = torch.randn((batch, latent), device='cuda', dtype=torch.float32)
    weight = torch.randn((vocab, latent), device='cuda', dtype=torch.float32)
    t = torch.randint(0, vocab, (batch,), device='cuda', dtype=torch.int32)
    
    acc = compute_accuracy(x, weight, t)
    print(f"Accuracy: {acc:.4f}")
